# Project: "PlantMate" - A Grounded Q&A Assistant over Real Company Documents

**Course project | Retrieval Augmented Generation (RAG)**

---

## 1. Business context

NorthWind Energy Services runs process plants and deputes engineers to client
sites. Its policies live in Word documents on a shared drive: an **HSE Policy
Manual**, an **Employee Leave and Business Travel Policy**, and a **Plant
Turnaround Maintenance SOP**.

Every week the HSE desk, HR helpdesk and maintenance planners answer the same
questions by hand:

- *"Is my hot work permit still valid after shift change?"*
- *"How much per diem do I get in Mumbai?"*
- *"Can we add this job after scope freeze?"*

A plain LLM cannot answer these - the policies are internal and were never in
its training data. If you ask anyway, it will either refuse or, worse,
**confidently invent a number**. In a plant, an invented safety limit is a
serious problem.

## 2. Your objective

Build **PlantMate**: a retrieval-augmented assistant that answers questions
**only** from these Word documents, **cites the document and section** it used,
and **refuses** when the answer is not in the corpus.


## 3. Deliverables

| # | Deliverable | Notes |
|---|---|---|
| D1 | A working ingestion pipeline `.docx -> text -> chunks` | Must preserve heading structure |
| D2 | A ChromaDB collection with embeddings + metadata | Metadata: source file, section, chunk index |
| D3 | A `search()` function with top-k and metadata filtering | Filter by document must work |
| D4 | An `ask()` function that answers **with citations** | Must say "I don't know" when unsupported |
| D5 | An evaluation run over the supplied `EVAL_SET` | Report retrieval hit-rate, answer accuracy, refusal accuracy |
| D6 | A short written analysis (markdown cells) | What failed, what you tuned, what you would do next |

## 4. Evaluation rubric (100 marks)

| Criterion | Marks |
|---|---|
| Ingestion: real `.docx` parsed, headings preserved, chunking sensible | 15 |
| Chunking quality: size + overlap justified, no mid-sentence damage | 15 |
| Vector store: correct embeddings, metadata, cosine space | 10 |
| Retrieval: top-k works, metadata filter works, scores inspected | 15 |
| Generation: grounded prompt, citations present, refusal behaviour correct | 20 |
| Evaluation: all 8 eval questions run, metrics reported honestly | 15 |
| Analysis and hygiene: no hard-coded API key, clear commentary | 10 |

## 5. Ground rules

1. **No hard-coded API keys.** Use `getpass` / environment variables. A key
   pasted in a notebook is a leaked key.
2. The assistant answers **only** from the documents. No outside knowledge.
3. Every answer must carry a citation like `[NorthWind_HSE_Policy_Manual.docx > 2. Permit to Work (PTW) System]`.
4. If retrieval returns nothing relevant, the correct answer is
   *"I don't know based on the documents provided."*


---
# Step 0 - Environment and corpus

In [ ]:
# Colab / fresh environment setup
!pip install -q openai chromadb python-docx tiktoken

In [ ]:
# ---------------------------------------------------------------------------
# API KEY - NEVER hard-code a key inside a notebook you will share.
# A key pasted into a shared .ipynb is a leaked key and must be revoked.
# ---------------------------------------------------------------------------
import os, getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
print("client ready ->", CHAT_MODEL, "|", EMBEDDING_MODEL)

### Build (or bring) the document corpus

Run the next cell to generate the three Word documents. They are deliberately
written the way real corporate policies are written - document numbers,
revision dates, numbered sections, specific numeric limits.

**Bringing your own documents is encouraged.** Drop any `.docx` files into a
folder named `company_docs` and skip the generator.

In [ ]:
# ============================================================================
# Builds the "real world" Word document corpus for this project.
# These are realistic corporate documents for a fictitious company,
# NorthWind Energy Services - an industrial / process plant services firm.
#
# >>> If you have your OWN .docx files, skip this cell, create a folder named
# >>> "company_docs" and drop your files in it. Everything downstream works
# >>> unchanged as long as your documents use real Word Heading styles.
# ============================================================================
import os
from docx import Document
from docx.shared import Pt

DOCS_DIR = "company_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

CORPUS = {
    'NorthWind_HSE_Policy_Manual.docx': [
        ('h1', 'NorthWind Energy Services - HSE Policy Manual'),
        ('p', 'Document No: HSE-MAN-001 | Revision: 4.2 | Effective Date: 01 April 2025 | Document Owner: Corporate HSE Department | Next Review: 01 April 2027'),
        ('h2', '1. Purpose and Scope'),
        ('p', 'This manual defines the minimum Health, Safety and Environment (HSE) requirements applicable to all NorthWind Energy Services employees, contractors, sub-contractors, trainees and visitors at every operating plant, project site, workshop and corporate office.'),
        ('p', 'Where a client HSE standard is more stringent than this manual, the client standard shall prevail. Any deviation from this manual requires written approval from the Corporate HSE Head.'),
        ('h2', '2. Permit to Work (PTW) System'),
        ('p', 'No non-routine work shall commence in a process area without a valid Permit to Work signed by the Area Authority and countersigned by the Performing Authority.'),
        ('p', 'A Hot Work Permit is valid for a maximum of 8 hours and for one shift only. It cannot be extended across shifts and must be re-issued after a shift change.'),
        ('p', 'A Confined Space Entry Permit is valid for a maximum of 12 hours. Entry is allowed only after a gas test confirms oxygen between 19.5% and 23.5% by volume, hydrocarbon below 5% LEL, and hydrogen sulphide below 5 ppm.'),
        ('p', 'Gas testing inside a confined space shall be repeated every 2 hours by a certified gas tester, and after any break in work exceeding 30 minutes. A trained standby person shall remain at the manway for the full duration of the entry.'),
        ('h2', '3. Personal Protective Equipment (PPE)'),
        ('p', 'The minimum PPE for entry into any process area is: flame retardant (FR) coverall, safety helmet with chin strap, safety goggles, safety shoes with steel toe cap, and hand gloves appropriate to the task.'),
        ('p', 'In designated H2S zones, a personal gas detector and an escape respirator (minimum 10 minutes rated) shall be carried at all times. Personnel working in H2S zones must hold a valid H2S awareness certificate renewed every 2 years.'),
        ('p', 'Face shields are mandatory for chemical handling, grinding and line breaking activities. Fall protection with a full body harness and double lanyard is mandatory for any work above 1.8 metres.'),
        ('h2', '4. Lockout Tagout (LOTO) and Energy Isolation'),
        ('p', 'Every energy isolation point shall carry an individual personal lock and a danger tag bearing the name, employee number and contact number of the person applying it.'),
        ('p', 'Where multiple trades work on the same equipment, a group lockout box shall be used. Each person places a personal lock on the group box and removes it only after their own work is complete.'),
        ('p', 'Only the person who applied a lock is authorised to remove it. Unauthorised removal of a lock or tag is treated as a Category 1 safety violation, resulting in disciplinary action and withdrawal of the site gate pass.'),
        ('h2', '5. Incident Reporting and Investigation'),
        ('p', 'All incidents, including near misses, first aid cases and process upsets, shall be reported verbally to the Shift In-Charge immediately and logged in the EHS Portal within 24 hours of occurrence.'),
        ('p', 'A Tier-1 process safety event, a lost time injury, or any fire requiring plant emergency response shall be notified to the Plant Manager and Corporate HSE within 2 hours.'),
        ('p', 'Root cause analysis for a Tier-1 event shall be completed within 15 working days using the 5-Why and Fishbone methodology, and shall be led by an investigator independent of the affected department.'),
        ('p', 'Each employee is expected to report a minimum of 5 near misses or unsafe conditions per calendar year. Reporting a near miss shall never attract punitive action.'),
        ('h2', '6. Emergency Response and Evacuation'),
        ('p', 'Plant alarm codes are as follows: a continuous siren of 2 minutes indicates a general emergency; an intermittent siren indicates a fire; a single continuous siren of 1 minute indicates all clear.'),
        ('p', 'On hearing the general emergency alarm, all personnel shall stop work, make equipment safe if it is safe to do so, and proceed upwind to the nearest assembly point A, B or C as marked on the plant layout.'),
        ('p', 'Head count at the assembly point shall be completed and reported to the Incident Controller within 10 minutes of the alarm. No person shall leave the assembly point without permission from the Assembly Point Warden.'),
        ('h2', '7. Contractor HSE Management'),
        ('p', 'Every contractor employee shall complete an 8-hour site HSE induction before being issued a gate pass. The induction is valid for 12 months and must be refreshed thereafter.'),
        ('p', 'Contractor HSE performance is scored monthly on leading and lagging indicators. A contractor scoring below 70% for two consecutive months shall be suspended from site until a corrective action plan is accepted.'),
        ('h2', '8. Environmental Compliance'),
        ('p', 'Stack emission monitoring shall be carried out monthly by an NABL accredited laboratory and the results submitted to the State Pollution Control Board quarterly.'),
        ('p', 'Treated effluent shall be maintained within a pH range of 6.5 to 8.5 before discharge. Any excursion beyond this range must be reported to the Environment Cell within 4 hours.'),
        ('p', 'Hazardous waste shall be stored in the designated covered yard for no longer than 90 days and disposed only through a CPCB authorised handler with a valid manifest.'),
    ],
    'NorthWind_Leave_and_Travel_Policy.docx': [
        ('h1', 'NorthWind Energy Services - Employee Leave and Business Travel Policy'),
        ('p', 'Document No: HR-POL-007 | Revision: 3.0 | Effective Date: 01 January 2025 | Document Owner: HR Operations | Applicable Payroll: India'),
        ('h2', '1. Applicability'),
        ('p', 'This policy applies to all confirmed full-time employees on the India payroll. Interns, retainers and third-party contract staff are governed by their individual engagement letters and are not covered by this policy.'),
        ('h2', '2. Leave Entitlements'),
        ('p', 'Earned Leave (EL): 21 working days per calendar year, accrued at 1.75 days per completed month of service. A maximum of 45 days of EL may be carried forward; any balance above 45 days lapses on 31 December. Accumulated EL is encashed at basic salary at the time of separation.'),
        ('p', 'Casual Leave (CL): 7 days per calendar year, to be availed for a maximum of 3 consecutive days at a time. Casual Leave cannot be carried forward and cannot be encashed.'),
        ('p', "Sick Leave (SL): 10 days per calendar year. A registered medical practitioner's certificate is mandatory for any sick leave exceeding 3 consecutive days."),
        ('p', 'Maternity Leave: 26 weeks of paid leave for the first two children and 12 weeks for the third child onwards, in line with the Maternity Benefit Act.'),
        ('p', 'Paternity Leave: 10 working days, to be availed within 6 months of the date of birth or adoption.'),
        ('p', 'Bereavement Leave: 5 calendar days in the event of the death of an immediate family member, defined as spouse, child, parent, sibling or parent-in-law.'),
        ('h2', '3. Leave Application and Approval'),
        ('p', 'All leave shall be applied through the Workday self-service portal. Earned Leave of more than 2 days must be applied at least 7 calendar days in advance.'),
        ('p', 'Emergency leave may be intimated by telephone or email to the reporting manager and must be regularised in Workday within 48 hours of resuming duty.'),
        ('p', 'Leave of up to 10 consecutive days is approved by the reporting manager. Any leave exceeding 10 consecutive days additionally requires Function Head approval.'),
        ('p', 'Leave without pay is permitted only after all leave balances are exhausted and requires HR Head approval.'),
        ('h2', '4. Site Deputation and Shift Allowances'),
        ('p', 'Employees deputed to a client site for a continuous period exceeding 15 days are eligible for a site deputation allowance of INR 1,200 per day from day 16 onwards.'),
        ('p', 'A night shift allowance of INR 450 per shift is payable for any shift where more than 4 hours fall between 22:00 and 06:00.'),
        ('p', 'Deputation allowance and per diem are not payable simultaneously for the same day; the higher of the two shall apply.'),
        ('h2', '5. Business Travel Entitlements'),
        ('p', 'Domestic air travel in economy class is permitted where the equivalent train journey exceeds 6 hours or the total door-to-door travel time exceeds 8 hours. In all other cases, AC 2-tier train travel is the default mode.'),
        ('p', 'Employees at Grade M3 and above may travel business class on international flights with a scheduled flying time exceeding 8 hours. All other international travel is in economy class.'),
        ('p', 'Hotel ceilings excluding taxes are INR 6,500 per night in metro cities and INR 4,500 per night in non-metro cities. Where a company or client guesthouse is available at the location, its use is mandatory.'),
        ('p', 'Airport transfers may be claimed on actuals against receipts. Use of the company travel desk for all bookings is mandatory; self-booked tickets require prior written approval.'),
        ('h2', '6. Per Diem'),
        ('p', 'Domestic per diem is INR 1,500 per day for metro cities and INR 1,100 per day for non-metro cities, payable without receipts to cover meals and incidentals.'),
        ('p', 'International per diem is USD 65 per day. Where meals are provided by the client or included in the hotel tariff, only 50% of the applicable per diem is payable.'),
        ('p', 'Per diem for the day of departure and the day of return is paid at 50% where the travel duration on that day is less than 8 hours.'),
        ('h2', '7. Expense Claims and Reimbursement'),
        ('p', 'All travel expense claims shall be submitted in Concur within 21 days of trip completion. Claims submitted after 45 days require written CFO approval and may be rejected.'),
        ('p', 'An original GST compliant invoice is mandatory for any single expense above INR 5,000. Expenses without supporting documents are limited to INR 500 per trip.'),
        ('p', 'Reimbursement is processed in the payroll cycle following claim approval, typically within 15 working days of manager approval.'),
        ('h2', '8. International Deputation and Visa'),
        ('p', 'Passport validity of at least 6 months beyond the intended date of return is mandatory before any international travel booking is confirmed.'),
        ('p', 'Corporate travel insurance is arranged by HR Operations for every international trip and is mandatory. Travel without an active policy number is not permitted.'),
        ('p', 'Any international deputation exceeding 90 days triggers a mandatory tax equalisation review by the Finance department before mobilisation.'),
    ],
    'NorthWind_Turnaround_Maintenance_SOP.docx': [
        ('h1', 'NorthWind Energy Services - Plant Turnaround and Shutdown Maintenance SOP'),
        ('p', 'Document No: SOP-MNT-014 | Revision: 2.1 | Effective Date: 15 June 2025 | Document Owner: Maintenance Head | Review Frequency: Every 2 years'),
        ('h2', '1. Purpose'),
        ('p', 'This Standard Operating Procedure defines the planning, execution, start-up and close-out requirements for a planned plant turnaround (TA) or major shutdown, to deliver the scope safely, on schedule and within approved cost.'),
        ('h2', '2. Turnaround Phases and Milestones'),
        ('p', 'Phase 1 - Long Range Planning begins at T minus 12 months and produces the preliminary scope register, budget estimate and TA organisation chart.'),
        ('p', 'Phase 2 - Detailed Planning begins at T minus 6 months and produces the level-3 schedule, resource histogram, long lead material order and contractor award.'),
        ('p', 'Phase 3 - Pre-Turnaround Execution begins at T minus 3 months and covers scaffolding, insulation removal, temporary facilities and pre-fabrication.'),
        ('p', 'Phase 4 - Execution covers shutdown, maintenance work and start-up. Phase 5 - Close-out is completed within 45 days of plant handover to Operations.'),
        ('h2', '3. Scope Management and Scope Freeze'),
        ('p', 'The turnaround scope shall be frozen at T minus 90 days. After the freeze date, no new scope may be added without written approval from the Turnaround Steering Committee.'),
        ('p', 'Any late scope addition estimated above 500 man-hours additionally requires joint approval from the Plant Manager and the Business Head, along with a schedule impact assessment.'),
        ('p', 'Every scope item shall carry a unique work order number, a justification category (statutory, reliability, or improvement) and an estimated man-hour value.'),
        ('h2', '4. Shutdown and Equipment Handover Sequence'),
        ('p', 'Throughput shall first be reduced to 50% of design capacity in a controlled manner, followed by feed cut as per the operations shutdown checklist.'),
        ('p', 'Equipment shall be depressurised to 0.5 barg, drained and purged with nitrogen until the hydrocarbon concentration is below 1% LEL, verified by two independent gas tests.'),
        ('p', 'Steam-out and water flushing shall follow purging where the service requires it. Isolation shall be completed strictly as per the approved blind list, and every blind installed shall be recorded in the blind register.'),
        ('p', 'Equipment is handed over to Maintenance only against a signed Equipment Handover Certificate issued by the Shift In-Charge, confirming isolation, de-energisation and gas-free status.'),
        ('h2', '5. Execution Control and Daily Reporting'),
        ('p', 'A daily turnaround coordination meeting shall be held at 07:00 hours, attended by the TA Manager, area coordinators, contractor supervisors and the HSE lead.'),
        ('p', 'Progress shall be tracked against a planned versus actual S-curve updated daily by 18:00 hours. Any critical path activity with float below 24 hours shall be escalated to the Turnaround Manager the same day.'),
        ('p', 'A recovery plan is mandatory when cumulative progress slips more than 5% behind plan.'),
        ('h2', '6. Quality Control and Inspection'),
        ('p', 'Hydrotest shall be carried out at 1.5 times the design pressure and held for a minimum of 30 minutes with no visible leakage or pressure drop.'),
        ('p', 'Positive Material Identification (PMI) is mandatory on 100% of alloy piping components before installation.'),
        ('p', 'Radiography coverage shall be 10% random for carbon steel welds and 100% for alloy welds. Any weld repair rate exceeding 5% triggers a welder qualification review.'),
        ('h2', '7. Start-up and Pre-Start-up Safety Review'),
        ('p', 'A Pre-Start-up Safety Review (PSSR) is mandatory before the introduction of hydrocarbons into any system, and shall be signed off by Operations, Maintenance, Inspection and HSE.'),
        ('p', 'All Punch List A items, defined as items that prevent safe start-up, must be closed to zero before PSSR sign-off. Punch List B items shall be closed within 30 days of start-up.'),
        ('p', 'Leak test at operating pressure shall be performed with nitrogen or with the process fluid at reduced pressure before ramping to full throughput.'),
        ('h2', '8. Close-out and Lessons Learned'),
        ('p', 'The turnaround close-out report shall be issued within 45 days of plant handover and shall include cost, schedule, HSE and quality performance against plan.'),
        ('p', 'A cost variance exceeding 10% of the approved budget requires a written explanation to the Business Head and the Finance Controller.'),
        ('p', 'Lessons learned shall be captured in a structured workshop within 30 days of start-up and uploaded to the corporate maintenance knowledge repository for reference in the next turnaround cycle.'),
    ],
}

for filename, blocks in CORPUS.items():
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)
    for kind, text in blocks:
        if kind == "h1":
            doc.add_heading(text, level=1)
        elif kind == "h2":
            doc.add_heading(text, level=2)
        else:
            doc.add_paragraph(text)
    doc.save(os.path.join(DOCS_DIR, filename))
    print("written:", filename)

print()
print("Corpus ready in ./%s" % DOCS_DIR)

### Sanity check - what a plain LLM does without RAG

Run this and **keep the output**. Your report must contrast these answers with
your final RAG answers.

In [ ]:
# Baseline: ask the model WITHOUT any documents.
# This is the "why do we need RAG at all" cell - keep the output, you will
# contrast it with your RAG answer later.
baseline_questions = [
    "How long is a hot work permit valid at our plant?",
    "How many earned leave days do I get per year and how many can I carry forward?",
    "When is the turnaround scope freeze and who approves late additions?",
]

for q in baseline_questions:
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": q}],
    )
    print("Q:", q)
    print("A:", r.choices[0].message.content.strip()[:400])
    print("-" * 90)

---
# Task 1 - Load the Word documents (D1)

Write `load_docx(path)` that opens a `.docx` and returns plain text where
**heading structure is preserved as markdown markers**, because your chunker
will use those markers to decide where a section starts.

- A `Heading 1` paragraph becomes `# <text>`
- A `Heading 2` paragraph becomes `## <text>`
- Everything else stays as a normal line

Then write `load_corpus(folder)` returning a list of
`{"title": <filename>, "text": <full text>}`.

**Checkpoint:** print the number of documents and the character count of each.

In [ ]:
from docx import Document

def load_docx(path):
    """Return the document text with Heading 1/2 converted to '#' / '##' markers."""
    # TODO:
    #   doc = Document(path)
    #   for para in doc.paragraphs:
    #       inspect para.style.name  -> "Heading 1", "Heading 2", "Normal", ...
    #       build the marked-up line, skip empty paragraphs
    #   return "\n".join(lines)
    raise NotImplementedError


def load_corpus(folder="company_docs"):
    """Return [{'title': filename, 'text': ...}, ...] for every .docx in folder."""
    # TODO
    raise NotImplementedError


documents = load_corpus()
print("documents:", len(documents))
for d in documents:
    print(d["title"], "->", len(d["text"]), "characters")

---
# Task 2 - Chunk the documents (D2)

Write `chunk_document(doc, max_chars=700, overlap=120)`.

Requirements:

1. Start a **new chunk at every heading** - a chunk must never straddle two sections.
2. Split long sections at roughly `max_chars`, **breaking on paragraph or sentence
   boundaries**, never in the middle of a word.
3. Carry `overlap` characters from the end of the previous chunk into the next one,
   so a fact sitting on a chunk boundary is not lost.
4. Every chunk carries metadata: `id`, `source` (file name), `section` (heading text),
   `chunk_index`.

**Think before you code:** the class demo used 350 characters and no overlap.
Look at the policy documents - a single clause like the confined space gas limits
runs longer than that. Decide your size and **write one markdown line justifying it**.

**Checkpoint:** total chunk count, min/median/max chunk length, and a printout of
3 chunks from different sections.

In [ ]:
def chunk_document(doc, max_chars=700, overlap=120):
    """Heading-aware chunking with character overlap. Returns list of dicts."""
    # TODO: return [{'id':..., 'source':..., 'section':..., 'chunk_index':..., 'text':...}, ...]
    raise NotImplementedError


chunks = []
for d in documents:
    chunks.extend(chunk_document(d))

print("total chunks:", len(chunks))
# TODO: print min / median / max chunk length and inspect 3 sample chunks

---
# Task 3 - Embed and store in ChromaDB (D2)

1. Write `embed_texts(texts)` that calls the embeddings API. **Batch it** -
   send at most ~100 texts per request instead of one call per chunk.
2. Create a Chroma collection named `plantmate` with **cosine** distance.
3. Add ids, embeddings, documents and metadata.

**Watch out:** if you re-run the cell, Chroma will complain about duplicate ids
or silently keep stale vectors. Handle that (delete and recreate the collection).

**Checkpoint:** `collection.count()` equals `len(chunks)`.

In [ ]:
import chromadb

def embed_texts(texts, batch_size=100):
    """Return a list of embedding vectors, batched."""
    # TODO
    raise NotImplementedError


chroma_client = chromadb.Client()
# TODO: delete the collection if it already exists, then create it with
#       metadata={"hnsw:space": "cosine"} and add all chunks.

---
# Task 4 - Retrieval (D3)

Write `search(question, n=4, source=None, min_score=None)`:

- embeds the question, queries Chroma, returns
  `[{'source','section','text','score'}, ...]`
- `score` is a **similarity** (1 - cosine distance), not a distance
- `source` filters to a single document using a Chroma `where` clause
- `min_score` drops weak hits - this is what makes refusal possible later

**Investigate (write your findings in a markdown cell):**

- Run *"What PPE do I need in an H2S area?"* and look at the scores.
- Run the out-of-scope question *"What is the company policy on cryptocurrency trading?"*
  Note what the top score is. Chroma always returns `n` results, even when nothing
  is relevant - **that number is how you choose `min_score`**.

In [ ]:
def search(question, n=4, source=None, min_score=None):
    # TODO
    raise NotImplementedError


for q in ["What PPE do I need in an H2S area?",
          "What is the company policy on cryptocurrency trading?"]:
    print("Q:", q)
    for hit in search(q, n=4):
        print("  %.3f | %s > %s" % (hit["score"], hit["source"], hit["section"]))
    print("-" * 90)

---
# Task 5 - Grounded generation with citations (D4)

Write `ask(question, n=4, source=None, min_score=0.25)` that:

1. retrieves chunks,
2. **short-circuits to a refusal** if nothing clears `min_score` - do not spend a
   token on the LLM when retrieval has already failed,
3. builds a context block where each chunk is numbered and labelled with its
   source and section,
4. calls the chat model with `temperature=0` and a system prompt that forbids
   outside knowledge and **requires a citation**,
5. returns both the answer text and the hits used, so you can audit it.

Your system prompt must enforce:

- answer only from the numbered context,
- cite as `[source > section]` after each claim,
- if the context does not contain the answer, reply exactly
  `I don't know based on the documents provided.`,
- never guess a number.

In [ ]:
SYSTEM_PROMPT = """TODO - write the grounding rules here"""

def ask(question, n=4, source=None, min_score=0.25, show_context=False):
    # TODO
    raise NotImplementedError


ask("What are the gas test limits for a confined space entry?", show_context=True)

---
# Task 6 - Evaluate (D5)

Run the supplied `EVAL_SET` and report three metrics:

| Metric | Definition |
|---|---|
| **Retrieval hit rate** | fraction of answerable questions where `expect_source` appears in the top-k hits |
| **Answer accuracy** | fraction of answerable questions whose answer contains `expect_keyword` |
| **Refusal accuracy** | fraction of unanswerable questions the assistant correctly refused |

A keyword check is a crude proxy for correctness - say so in your analysis, and
mention what you would use instead in production (human review, or an LLM judge
scored against a reference answer).

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation set: 6 answerable questions + 2 deliberately unanswerable ones.
# "expect_source"  -> which document SHOULD be retrieved
# "expect_keyword" -> a string that must appear in a correct answer
# "answerable=False" -> the assistant MUST refuse instead of inventing an answer
# ---------------------------------------------------------------------------
EVAL_SET = [
    {"question": "What gas test readings are required before a confined space entry?",
     "expect_source": "NorthWind_HSE_Policy_Manual.docx", "expect_keyword": "19.5", "answerable": True},
    {"question": "How long is a hot work permit valid and can it be extended across shifts?",
     "expect_source": "NorthWind_HSE_Policy_Manual.docx", "expect_keyword": "8 hours", "answerable": True},
    {"question": "How many earned leave days do I get in a year and what is the carry forward limit?",
     "expect_source": "NorthWind_Leave_and_Travel_Policy.docx", "expect_keyword": "21", "answerable": True},
    {"question": "What is the per diem for domestic travel to a metro city?",
     "expect_source": "NorthWind_Leave_and_Travel_Policy.docx", "expect_keyword": "1,500", "answerable": True},
    {"question": "When is the turnaround scope frozen and who approves late scope additions?",
     "expect_source": "NorthWind_Turnaround_Maintenance_SOP.docx", "expect_keyword": "90", "answerable": True},
    {"question": "At what pressure is a hydrotest carried out and for how long is it held?",
     "expect_source": "NorthWind_Turnaround_Maintenance_SOP.docx", "expect_keyword": "1.5", "answerable": True},
    {"question": "What is the company policy on cryptocurrency trading by employees?",
     "expect_source": None, "expect_keyword": None, "answerable": False},
    {"question": "Who is the Chief Executive Officer of NorthWind Energy Services?",
     "expect_source": None, "expect_keyword": None, "answerable": False},
]

print(len(EVAL_SET), "evaluation questions loaded",
      "|", sum(1 for e in EVAL_SET if e["answerable"]), "answerable",
      "|", sum(1 for e in EVAL_SET if not e["answerable"]), "must-refuse")

In [ ]:
# TODO: loop over EVAL_SET, call search() and ask(), and print a results table
#       plus the three headline metrics.

---
# Task 7 - Written analysis (D6)

In markdown cells, answer:

1. **Chunk size.** What did you settle on and why? Rebuild the index with the
   350-character no-overlap scheme from the live class, re-run the evaluation, and
   report the difference. If a question flips from wrong to right, name it and
   explain what the chunker did to it.
2. **Threshold.** How did you pick `min_score`? What breaks if it is too high?
   Too low?
3. **Failures.** Which eval question performed worst? Read the retrieved chunks
   and diagnose it: retrieval problem or generation problem? They need different fixes.
4. **Before / after.** Contrast the Step 0 no-RAG answers with your RAG answers.
5. **Production.** Name three things you would add before letting real employees
   use this (think: document updates, access control, logging, evaluation cadence).

---

# Bonus (optional, not marked)

- **Hybrid search** - combine vector similarity with a keyword match so exact
  terms like `SOP-MNT-014` or `PSSR` are never missed.
- **Query rewriting** - expand `"perm valid?"` into a full question before embedding.
- **Conversation memory** - make follow-ups like *"and for non-metro?"* work.
- **Freshness metadata** - store the revision and effective date per document, and
  make the assistant state which revision it answered from.
- **Streamlit / Gradio UI** with a document filter dropdown.

---

# Submission checklist

- [ ] Notebook runs top to bottom on a clean runtime
- [ ] No API key visible anywhere in the file
- [ ] All 8 eval questions executed with visible output
- [ ] Three metrics reported
- [ ] Analysis cells written
